# Sprint 4 Experiment — Nemotron-3-Ultra-550B (OpenRouter Free) × Gemini Failures

This notebook runs the 76 questions that Gemini 3.1 Flash-Lite failed on through
Nemotron-3-Ultra-550B (free tier via OpenRouter), with reasoning enabled.

| Setting | Value |
|---------|-------|
| Model | nvidia/nemotron-3-ultra-550b-a55b:free (OpenRouter) |
| Questions | gemini_simple_failures.csv (76 rows — Gemini WRONG + REFUSED + NO ANSWER) |
| Prompt | simple (zero-shot, same as Gemini baseline) |
| Reasoning | enabled (OpenRouter reasoning API) |
| Chunk size | 3000 (author default — fixed) |
| Chunk overlap | 300 (author default — fixed) |
| Top-K | 5 (author default — fixed) |
| Temperature | 0.0 (professor requirement — fixed) |
| Embedding | all-MiniLM-L6-v2 (local, free) |

## Cell 1 — Setup paths

In [1]:
import sys, os

project_root = '/Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026'
sprint4_root = os.path.join(project_root, 'Sprint 4')
sprint3_uda  = os.path.join(project_root, 'Sprint 3', 'UDA-Benchmark')

for p in [sprint4_root, sprint3_uda]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(sprint3_uda)  # UDA preprocess uses relative paths from this root
print(f'project_root : {project_root}')
print(f'sprint4_root : {sprint4_root}')
print(f'sprint3_uda  : {sprint3_uda}')
print(f'cwd          : {os.getcwd()}')

if not os.path.isdir(sprint3_uda):
    raise FileNotFoundError(f'sprint3_uda not found: {sprint3_uda}')

project_root : /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026
sprint4_root : /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 4
sprint3_uda  : /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark
cwd          : /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


## Cell 2 — Load Gemini failures

In [2]:
import pandas as pd

MODEL_KEY  = 'nemotron-openrouter'
PROMPT     = 'simple'

# Source: questions Gemini failed on (76 rows)
FAILURES_CSV = os.path.join(
    sprint4_root,
    'experiments/gemini-3.1-flash-lite/results/gemini_simple_failures.csv'
)
OUTPUT_DIR = os.path.join(sprint4_root, f'experiments/{MODEL_KEY}/results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Full combined CSV needed for doc_name lookup
ALL_QUESTIONS_CSV = os.path.join(sprint4_root, 'benchmark/questions/all_questions_combined.csv')

# PDF directory lookup
UDA_PDF_BASE = os.path.join(sprint3_uda, 'dataset/src_doc_files_example')
PDF_DIRS = {
    'music_structured': os.path.join(UDA_PDF_BASE, 'music_docs'),
    'tathybrid':        os.path.join(UDA_PDF_BASE, 'tat_docs'),
    'finhybrid':        os.path.join(UDA_PDF_BASE, 'fin_docs'),
    'nqtext':           os.path.join(UDA_PDF_BASE, 'wiki_nq_docs/pdfs'),
    'fetatab':          os.path.join(UDA_PDF_BASE, 'wiki_feta_docs/pdfs'),
    'papertab':         os.path.join(UDA_PDF_BASE, 'paper_docs'),
    'papertext':        os.path.join(UDA_PDF_BASE, 'paper_docs'),
}

# Load failures and enrich with doc_name from master CSV
df_fail = pd.read_csv(FAILURES_CSV)
df_all  = pd.read_csv(ALL_QUESTIONS_CSV)

# Merge doc_name in (failures CSV doesn't have it)
df_fail = df_fail.merge(
    df_all[['question_id', 'doc_name']],
    on='question_id',
    how='left'
)

# Rename ground_truth column to match RAGRunner expectation
if 'ground_truth' not in df_fail.columns and 'llm_answer' in df_fail.columns:
    df_fail = df_fail.rename(columns={'llm_answer': 'gemini_answer'})

print(f'Gemini failures loaded: {len(df_fail)} questions')
print()
print('By dataset:')
print(df_fail['dataset'].value_counts().to_string())
print()
print('By failure_type:')
print(df_fail['failure_type'].value_counts().to_string())
print()

# Verify PDFs
print('PDF availability check:')
missing = []
for _, row in df_fail.iterrows():
    pdf_dir = PDF_DIRS.get(row['dataset'], '')
    pdf_path = os.path.join(pdf_dir, str(row['doc_name']) + '.pdf')
    if not os.path.exists(pdf_path):
        missing.append(f"{row['dataset']}/{row['doc_name']}")
missing = list(dict.fromkeys(missing))
if missing:
    print(f'  WARNING — {len(missing)} PDFs not found:')
    for p in missing: print(f'    {p}')
else:
    print(f'  All PDFs found.')

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/I772947/Library/Python/3.9/lib/python/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/I772947/Library/Python/3.9/lib/python/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/I772947/Library/Python/3.9/lib/python/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/Users/I772947/Library/Python/3.9/lib/python/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
  File "/Users/I772947/Library/Python/3.9/lib/python/site-packages/traitlets/traitlets.py", line 668, in get
    value = self._validate(obj, default)
  File

Gemini failures loaded: 76 questions

By dataset:
dataset
finhybrid           31
tathybrid           29
music_structured    16

By failure_type:
failure_type
REFUSED      38
WRONG        37
NO ANSWER     1

PDF availability check:
  All PDFs found.


## Cell 3 — Run Nemotron on Gemini failures

In [3]:
from framework.rag_runner import RAGRunner
from datetime import datetime

all_results = []

for dataset, group_df in df_fail.groupby('dataset'):
    print(f"\n{'='*60}")
    print(f'Dataset: {dataset}  ({len(group_df)} questions)')
    print(f"{'='*60}")

    pdf_dir = PDF_DIRS.get(dataset, '')
    if not pdf_dir or not os.path.isdir(pdf_dir):
        print(f'  SKIP — PDF directory not found: {pdf_dir}')
        continue

    # Save temp CSV for RAGRunner.run()
    tmp_csv = os.path.join(OUTPUT_DIR, f'_tmp_{dataset}.csv')
    group_df.to_csv(tmp_csv, index=False)

    # music_structured has no UDA eval metric — use nqtext as neutral runner
    runner_dataset = dataset if dataset != 'music_structured' else 'nqtext'

    runner = RAGRunner(model_key=MODEL_KEY, dataset=runner_dataset, prompt=PROMPT)
    results_df = runner.run(
        questions_csv=tmp_csv,
        pdf_dir=pdf_dir,
        doc_col='doc_name',
        output_dir=OUTPUT_DIR,
    )
    results_df['dataset_actual'] = dataset
    all_results.append(results_df)
    os.remove(tmp_csv)

# Combine all datasets
results_combined = pd.concat(all_results, ignore_index=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
final_path = os.path.join(OUTPUT_DIR, f'ALL_RESULTS_{MODEL_KEY}_{PROMPT}_{ts}.csv')
results_combined.to_csv(final_path, index=False)
print(f'\nAll results saved → {final_path}')
print(f'Total rows: {len(results_combined)}')

/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/I772947/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/I772947/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and t


Dataset: finhybrid  (31 questions)


/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAGRunner ready: model=nemotron-openrouter, dataset=finhybrid, prompt=simple
  CHUNK_SIZE=3000, CHUNK_OVERLAP=300, TOP_K=5, TEMP=0.0

  PDF: ABMD_2012.pdf
  110 chunks indexed
  [1/7] during the 2012 year , did the equity awards in which the prescribed p...
    → 
  [2/7] for equity awards where the performance criteria has been met in 2012 ...
    → The user is asking for the average compensation expense per year over which the 
  [3/7] did abiomed outperform the nasdaq medical equipment index?...
    → The provided context does not include Abiomed's stock performance data to compar
  [4/7] did abiomed outperform the nasdaq composite index?...
    → The user is asking whether Abiomed outperformed the Nasdaq Composite Index based
  [5/7] how much of total future minimum lease payments are due currently?...
    → The answer is: $1,473,000
  [6/7] what is the roi of an investment in abiomed inc from march 2007 to mar...
    → The user is asking for the ROI (Return on Investment) of an in

## Cell 4 — Compare Nemotron vs Gemini (side by side)

In [4]:
# Side-by-side comparison: for each question, show Gemini failure vs Nemotron answer
comparison = results_combined[['question_id', 'dataset_actual', 'question', 'ground_truth', 'response']].copy()
comparison = comparison.rename(columns={'response': 'nemotron_answer', 'dataset_actual': 'dataset'})

# Merge Gemini's answer back in
gemini_col = 'llm_answer' if 'llm_answer' in df_fail.columns else 'gemini_answer'
comparison = comparison.merge(
    df_fail[['question_id', gemini_col, 'failure_type']].rename(columns={gemini_col: 'gemini_answer'}),
    on='question_id', how='left'
)

# Quick empty-rate check
comparison['nemotron_empty'] = comparison['nemotron_answer'].fillna('').str.strip() == ''
print('=== Nemotron answer rate by dataset ===')
summary = comparison.groupby('dataset').agg(
    total=('question_id','count'),
    answered=('nemotron_empty', lambda x: (~x).sum()),
    empty=('nemotron_empty','sum'),
).reset_index()
summary['answer_rate'] = (summary['answered'] / summary['total'] * 100).round(1)
print(summary.to_string(index=False))

# Save comparison
cmp_path = os.path.join(OUTPUT_DIR, f'comparison_gemini_vs_nemotron_{ts}.csv')
comparison.to_csv(cmp_path, index=False)
print(f'\nComparison saved → {cmp_path}')

=== Nemotron answer rate by dataset ===
         dataset  total  answered  empty  answer_rate
       finhybrid     31        30      1         96.8
music_structured     16        16      0        100.0
       tathybrid     29        29      0        100.0

Comparison saved → /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 4/experiments/nemotron-openrouter/results/comparison_gemini_vs_nemotron_20260720_115937.csv


## Cell 5 — Sample answers (spot check)

In [5]:
# Show first 5 questions with ground truth, Gemini failure, and Nemotron answer
pd.set_option('display.max_colwidth', 120)
spot = comparison[['question_id', 'dataset', 'failure_type', 'question',
                    'ground_truth', 'gemini_answer', 'nemotron_answer']].head(10)
for _, row in spot.iterrows():
    print(f"\n--- {row['question_id']} [{row['dataset']}] [{row['failure_type']}] ---")
    print(f"Q  : {row['question']}")
    print(f"GT : {row['ground_truth']}")
    print(f"GEM: {str(row['gemini_answer'])[:150]}")
    print(f"NEM: {str(row['nemotron_answer'])[:150]}")


--- S3_FINHYBRID_ABMD/2012/page_75.pdf-1 [finhybrid] [WRONG] ---
Q  : during the 2012 year , did the equity awards in which the prescribed performance milestones were achieved exceed the equity award compensation expense for equity granted during the year?
GT : yes
GEM: The provided text states that for the year ended March 31, 2012, the Company recorded $3.3 million in stock-based compensation expense for equity awar
NEM: 

--- S3_FINHYBRID_ABMD/2012/page_75.pdf-2 [finhybrid] [WRONG] ---
Q  : for equity awards where the performance criteria has been met in 2012 , what is the average compensation expense per year over which the cost will be expensed?
GT : 1719526 | 1714285.71429
GEM: The provided text states that the remaining unrecognized compensation expense for equity awards as of March 31, 2012, is $3.6 million, and the weighte
NEM: The user is asking for the average compensation expense per year over which the cost will be expensed for equity awards where performance criteria has